In [1]:
!pip install gymnasium numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 965.4/965.4 kB 14.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [gymnasium]/3 [gymnasium]

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [2]:
import gymnasium as gym
import numpy as np
import random

# Create the Taxi-v3 environment
env = gym.make("Taxi-v3", render_mode="ansi")

# Reset the environment to generate the initial state
state, info = env.reset()

# Visualize the initial state of the environment
print(env.render())

+---------+
|R: | : :G|
| : | : : |
| : : : : |
| | : | : |
|Y| : |B: |
+---------+




In [3]:
# Get the number of states and actions
state_space = env.observation_space.n
action_space = env.action_space.n

print(f"There are {state_space} possible states.")
print(f"There are {action_space} possible actions.")

# Initialize the Q-table with zeros
q_table = np.zeros((state_space, action_space))
print("Q-Table initialized with shape:", q_table.shape)

There are 500 possible states.
There are 6 possible actions.
Q-Table initialized with shape: (500, 6)


In [4]:
# Training parameters
total_episodes = 25000        # Total number of training episodes
learning_rate = 0.7           # Learning rate (alpha)
max_steps = 99                # Max steps per episode
gamma = 0.95                  # Discounting rate (gamma)

# Exploration parameters for Epsilon-Greedy strategy
epsilon = 1.0                 # Starting exploration rate
max_epsilon = 1.0             # Maximum exploration probability
min_epsilon = 0.05            # Minimum exploration probability
decay_rate = 0.005            # Exponential decay rate for exploration

In [5]:
# ==========================================
# 1. TRAINING PHASE
# ==========================================
print("Training agent...")
for episode in range(total_episodes):
    # Reset the environment for a new episode
    state, info = env.reset()
    done = False

    for step in range(max_steps):
        # Epsilon-greedy policy: Decide to explore or exploit
        exp_exp_tradeoff = random.uniform(0, 1)

        if exp_exp_tradeoff > epsilon:
            # Exploit: Choose the action with the highest Q-value for the current state
            action = np.argmax(q_table[state, :])
        else:
            # Explore: Choose a random action
            action = env.action_space.sample()

        # Take the action and observe the outcome
        new_state, reward, terminated, truncated, info = env.step(action)
        done = terminated or truncated

        # Update the Q-table using the Bellman Equation
        q_table[state, action] = q_table[state, action] + learning_rate * (
            reward + gamma * np.max(q_table[new_state, :]) - q_table[state, action]
        )

        # Transition to the next state
        state = new_state

        # End episode if the agent dropped off the passenger successfully or hit the time limit
        if done:
            break

    # Decay the epsilon to reduce exploration over time
    epsilon = min_epsilon + (max_epsilon - min_epsilon) * np.exp(-decay_rate * episode)

print("Training Complete!")

# ==========================================
# 2. EVALUATION PHASE
# ==========================================
eval_episodes = 100
total_rewards = 0
total_steps = 0

print("\nEvaluating agent over 100 episodes...")
for episode in range(eval_episodes):
    state, info = env.reset()
    done = False
    episode_reward = 0

    for step in range(max_steps):
        # In evaluation, we always exploit our learned Q-table (no exploration)
        action = np.argmax(q_table[state, :])

        new_state, reward, terminated, truncated, info = env.step(action)
        done = terminated or truncated

        episode_reward += reward
        state = new_state
        total_steps += 1

        if done:
            break

    total_rewards += episode_reward

print(f"Average Reward per episode: {total_rewards / eval_episodes}")
print(f"Average Steps per episode: {total_steps / eval_episodes}")

# Close the environment
env.close()

Training agent...
Training Complete!

Evaluating agent over 100 episodes...
Average Reward per episode: 7.84
Average Steps per episode: 13.16
